In [1]:
import os
import sys
sys.path.append(os.path.abspath('../'))

In [2]:
import networkx as nx
from pprint import pprint

from utils.network import load_network


nsf = load_network('NSF')
print(f"Nodes: {nsf.number_of_nodes()}")
print(f"Edges: {nsf.number_of_edges()}")

Nodes: 14
Edges: 42


In [3]:
def length_path(graph: nx.DiGraph, physical_p: tuple[int]) -> float:
    p_length = 0
    Num_n_p = len(physical_p)
    for n_index in range(Num_n_p - 1):
        n1 = physical_p[n_index]
        n2 = physical_p[n_index + 1]
        e_length = graph[n1][n2]['weight']
        p_length += e_length
    return p_length

def wang_ksp(graph: nx.DiGraph, num_paths: int) -> dict[(int, int), list[tuple[int]]]:
    ksp_sd_set = {}
    nodes = list(graph.nodes)
    for s in nodes:
        for d in nodes:
            if s != d:
                ksp_sd = []
                all_physical_path_sd = sorted(list(nx.shortest_simple_paths(graph, s, d)), 
                                              key=lambda path: (len(path), length_path(graph, path)))
                for physical_path in all_physical_path_sd[:num_paths]:
                    ksp_sd.append(tuple(physical_path))
                ksp_sd_set[(s, d)] = ksp_sd

    return ksp_sd_set

wang = wang_ksp(nsf, 2)

In [4]:
from src.paths.algorithms.k_shortest_paths import KShortestPaths

ksp = KShortestPaths('NSF', 2, params={'path_weight': 'hop'})
mori = ksp.select_k_paths_all_pairs()
for key, paths in mori.items():
    tuple_path = []
    for path in paths:
        tuple_path.append(tuple(path))
    mori[key] = tuple_path

In [5]:
for (s, d), paths in wang.items():
    if paths == mori[(s, d)]:
        continue
    print(f"({s}, {d})")
    print("Wang:")
    pprint(paths)
    print("Mori:")
    pprint(mori[(s, d)])
    print()

(2, 9)
Wang:
[(2, 1, 8, 9), (2, 4, 11, 12, 9)]
Mori:
[(2, 1, 8, 9), (2, 3, 6, 10, 9)]

(4, 3)
Wang:
[(4, 2, 3), (4, 5, 6, 3)]
Mori:
[(4, 2, 3), (4, 2, 1, 3)]

